# TEM axon & myelin segmentation: full test-set inference

Runs the complete submitted pipeline on a folder of TEM micrographs and writes one instance mask per image.

1. 2×2 block-mean downsampling (4096×3008 → 2048×1504 px, ~4.30 nm/px).
2. 10-model nnU-Net ensemble (5 folds × 250 epochs + 5 folds × 500 epochs) with mirroring TTA.
3. Bilinear upsampling of the averaged probabilities to native resolution, P(myelin) × 0.7, argmax.
4. Axon-seeded watershed: one fiber per axon, encoded as axon `X` (even ≥ 2) and myelin `X+1`.
5. Removal of fibers whose axon touches the image border and of fibers without myelin.

**Usage:** set `IMAGES_DIR` in the next cell and run all cells (*Run → Run All Cells*).
The notebook first checks the paths, the model files and the libraries, then predicts every image,
saves its mask (`uint16` `.tif`, same file name as the input) and shows the image next to its prediction.

In [ ]:
from pathlib import Path

# ===================== CONFIGURACIÓN: solo hay que cambiar IMAGES_DIR =====================
# rutas relativas a la carpeta de este notebook (también se aceptan rutas absolutas)
IMAGES_DIR = Path("test_samples")       # carpeta con las imágenes de test (.tif / .png)
OUTPUT_DIR = Path("test_predictions")   # se crea automáticamente
# ==========================================================================================

CHECKPOINTS_DIR = Path("checkpoints")   # contiene model/ (250 épocas) y model_500ep/ (500 épocas)
GPU = "0"                               # GPU a usar; si no hay GPU se usa la CPU (mucho más lento)
SHOW_FIGURES = True                     # muestra cada imagen junto a su predicción
SAVE_FIGURES = False                    # True: guarda también esas figuras en OUTPUT_DIR/figures/

## 1. Paths

In [ ]:
EXTS = (".tif", ".tiff", ".png")

print(f"Carpeta de imágenes: {IMAGES_DIR.resolve()}")
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"  NO EXISTE. Revisa IMAGES_DIR en la celda de configuración: {IMAGES_DIR.resolve()}")
files = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in EXTS)
if not files:
    raise FileNotFoundError(f"  existe, pero no contiene imágenes {EXTS}")
print(f"  existe: {len(files)} imágenes ({files[0].name} ... {files[-1].name})")

print(f"\nCarpeta de salida: {OUTPUT_DIR.resolve()}")
if OUTPUT_DIR.is_dir():
    print("  ya existe (los archivos con el mismo nombre se sobrescriben)")
else:
    OUTPUT_DIR.mkdir(parents=True)
    print("  creada")

## 2. Model files

In [ ]:
MODEL_DIRS = [CHECKPOINTS_DIR / "model", CHECKPOINTS_DIR / "model_500ep"]
FOLDS = (0, 1, 2, 3, 4)
CHECKPOINT = "checkpoint_final.pth"

print(f"Carpeta de modelos: {CHECKPOINTS_DIR.resolve()}")
missing_files = []
for m in MODEL_DIRS:
    print(f"  {m.name}/")
    needed = [m / "plans.json", m / "dataset.json"] + [m / f"fold_{f}" / CHECKPOINT for f in FOLDS]
    for f in needed:
        ok = f.is_file()
        size = f"{f.stat().st_size / 2**20:8.1f} MB" if ok else ""
        print(f"    {'existe   ' if ok else 'NO EXISTE'}  {f.relative_to(m)}  {size}")
        if not ok:
            missing_files.append(f)
if missing_files:
    raise FileNotFoundError(f"Faltan {len(missing_files)} archivos de modelo. Revisa CHECKPOINTS_DIR "
                            "o descárgalos con: python download_weights.py")
print("todos los archivos de los modelos existen")

## 3. Libraries

In [ ]:
import importlib.util
import sys

# comprueba las librerías antes de importarlas y explica cómo instalar las que falten
REQUIRED = {  # módulo -> paquete de pip
    "numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib",
    "tifffile": "tifffile", "PIL": "Pillow", "skimage": "scikit-image",
    "torch": "torch", "nnunetv2": "nnunetv2==2.8.1",
}
missing = []
for mod, pkg in REQUIRED.items():
    ok = importlib.util.find_spec(mod) is not None
    print(f"  {'instalada   ' if ok else 'NO INSTALADA'}  {pkg}")
    if not ok:
        missing.append(pkg)
if missing:
    msg = [f"Faltan librerías en este kernel ({sys.executable}): {', '.join(missing)}", "",
           "Opciones para instalarlas:",
           "  a) en una celda de este notebook:",
           f"       %pip install {' '.join(p for p in missing if p != 'torch')}",
           "  b) en una terminal, con el entorno activado (versiones exactas):",
           "       pip install -r requirements.txt"]
    if "torch" in missing:
        msg += ["  PyTorch con soporte de GPU (CUDA 12.8) se instala aparte, antes que lo demás:",
                "       pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128"]
    msg += ["", "Después de instalar, reinicia el kernel (Kernel -> Restart) y vuelve a ejecutar todo.",
            "Si las librerías ya están instaladas en otro entorno, elige ese kernel (Kernel -> Change Kernel)."]
    raise ImportError("\n".join(msg))
print("todas las librerías están instaladas")

## 4. Setup

In [ ]:
import os

# deben fijarse antes de importar torch / nnU-Net
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.environ.setdefault(k, "")  # nnU-Net solo avisa si no existen; para inferir no se usan

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn.functional as F
from matplotlib.colors import to_rgb
from IPython.display import display
from matplotlib.patches import Patch
from PIL import Image
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# nnU-Net cambia el backend de matplotlib a 'agg' al importarse; se vuelve al de Jupyter para ver las figuras
%matplotlib inline

# parámetros del pipeline entregado (no cambiar para reproducir las predicciones)
SCALE = 2             # submuestreo 2×2
MYELIN_WEIGHT = 0.7   # factor sobre P(mielina) antes del argmax
STEP = 0.5            # solapamiento de las ventanas de nnU-Net

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")

## 5. Pre- and post-processing

In [ ]:
CONN8 = np.ones((3, 3), dtype=bool)


def read_image(p: Path) -> np.ndarray:
    img = tifffile.imread(p) if p.suffix.lower() in (".tif", ".tiff") else np.asarray(Image.open(p))
    return img[..., 0] if img.ndim == 3 else img  # por si viniera con canales repetidos


def write_label(p: Path, lbl: np.ndarray):
    lbl = lbl.astype(np.uint16)
    if p.suffix.lower() in (".tif", ".tiff"):
        tifffile.imwrite(p, lbl)
    else:
        Image.fromarray(lbl).save(p)


def downsample(img: np.ndarray, s: int = SCALE) -> np.ndarray:
    """Media de bloques s×s redondeada a uint8 (igual que en el entrenamiento)."""
    H, W = img.shape
    b = img[: H - H % s, : W - W % s].reshape(H // s, s, W // s, s).astype(np.float32)
    return np.round(b.mean(axis=(1, 3))).astype(np.uint8)


def semantic_to_instances(sem: np.ndarray) -> np.ndarray:
    """Semántica (0 fondo, 1 axón, 2 mielina) -> fibra k: axón 2k, mielina 2k+1.

    Una semilla por componente conexa de axón; watershed sobre la distancia al axón, restringido a
    axón ∪ mielina: cada píxel de mielina va a la fibra del axón más cercano.
    """
    axon = sem == 1
    fiber = axon | (sem == 2)
    seeds, n = ndi.label(axon, structure=CONN8)
    out = np.zeros(sem.shape, dtype=np.uint16)
    if n == 0:
        return out
    fibers = watershed(ndi.distance_transform_edt(~axon), markers=seeds, mask=fiber)
    out[axon] = 2 * fibers[axon]
    my = fiber & ~axon & (fibers > 0)
    out[my] = 2 * fibers[my] + 1
    return out


def filter_fibers(inst: np.ndarray) -> np.ndarray:
    """Quita las fibras que el protocolo de anotación no etiqueta y renumera 2k / 2k+1.

    - fibras cuyo AXÓN toca el borde de la imagen (en el GT son artefacto, ignorado)
    - fibras sin mielina
    """
    fid = np.where(inst >= 2, inst // 2, 0).astype(np.int64)
    n = int(fid.max()) + 1
    if n == 1:
        return inst
    axon_id = np.where((inst >= 2) & (inst % 2 == 0), fid, 0)
    remove = np.zeros(n, dtype=bool)
    edge = np.concatenate([axon_id[0], axon_id[-1], axon_id[:, 0], axon_id[:, -1]])
    remove[np.unique(edge)] = True
    area = np.bincount(fid.ravel(), minlength=n)
    my_area = np.bincount(fid[(inst >= 3) & (inst % 2 == 1)], minlength=n)
    remove |= (area > 0) & (my_area == 0)
    remove[0] = True
    keep = np.flatnonzero(~remove & (area > 0))
    new_id = np.zeros(n, dtype=np.int64)
    new_id[keep] = np.arange(1, len(keep) + 1)
    k = new_id[fid]
    return np.where(k > 0, 2 * k + (inst % 2 == 1), 0).astype(np.uint16)


def probs_to_label(probs: np.ndarray, shape) -> np.ndarray:
    """Probabilidades (3, 1, h, w) a ½ resolución -> instancias a resolución nativa."""
    p = torch.from_numpy(np.asarray(probs, dtype=np.float32)).to(device)[:, 0]
    p = F.interpolate(p[None], size=tuple(shape), mode="bilinear", align_corners=False)[0]
    p[2] *= MYELIN_WEIGHT
    sem = p.argmax(0).to(torch.uint8).cpu().numpy()
    inst = semantic_to_instances(sem)
    out = filter_fibers(inst)
    return out if out.any() else inst  # nunca se entrega una imagen vacía

## 6. Visualization

In [ ]:
AXON_COLOR, MYELIN_COLOR = "#2a78d6", "#eb6834"
BORDER_THICKNESS = 10       # grosor de los bordes en la superposición (px)
MYELIN_EDGE_THICKNESS = 6   # grosor del borde de mielina en el panel de instancias (px)
LABEL_FONTSIZE = 6


def fiber_stats(lbl: np.ndarray) -> tuple[int, float]:
    """Número de fibras y g-ratio mediano (g = sqrt(A_axón / A_fibra))."""
    fid = np.where(lbl >= 2, lbl // 2, 0).ravel()
    a_fib = np.bincount(fid)
    a_ax = np.bincount(fid[(lbl.ravel() % 2 == 0) & (fid > 0)], minlength=len(a_fib))
    ok = (a_fib > 0) & (np.arange(len(a_fib)) > 0)
    g = np.sqrt(a_ax[ok] / a_fib[ok])
    return int(ok.sum()), float(np.median(g)) if ok.any() else float("nan")


def boundaries(lbl: np.ndarray, thickness: int) -> np.ndarray:
    """Píxeles donde cambia la etiqueta respecto a un vecino, engrosados."""
    b = np.zeros(lbl.shape, dtype=bool)
    b[:-1] |= lbl[:-1] != lbl[1:]
    b[1:] |= lbl[1:] != lbl[:-1]
    b[:, :-1] |= lbl[:, :-1] != lbl[:, 1:]
    b[:, 1:] |= lbl[:, 1:] != lbl[:, :-1]
    if thickness > 1:
        b = ndi.binary_dilation(b, iterations=thickness - 1)
    return b & (lbl > 0)


def instances_to_rgb(lbl: np.ndarray, seed: int = 0) -> np.ndarray:
    """Un color por fibra: axón X en el tono claro y mielina X+1 en el mismo tono más oscuro."""
    fid = np.where(lbl >= 2, lbl // 2, 0)
    hues = np.random.default_rng(seed).uniform(0.25, 1.0, size=(fid.max() + 1, 3))
    hues[0] = 0
    rgb = hues[fid]
    rgb[(lbl >= 3) & (lbl % 2 == 1)] *= 0.55
    my_edge = boundaries(lbl, MYELIN_EDGE_THICKNESS) & (lbl % 2 == 1)
    rgb[my_edge] = 1.0
    return rgb


def overlay_rgba(lbl: np.ndarray) -> np.ndarray:
    """Bordes de axón (azul) y mielina (naranja) sobre fondo transparente."""
    edges = boundaries(lbl, BORDER_THICKNESS)
    rgba = np.zeros((*lbl.shape, 4), dtype=np.float32)
    rgba[edges & (lbl % 2 == 0)] = (*to_rgb(AXON_COLOR), 1.0)
    rgba[edges & (lbl % 2 == 1)] = (*to_rgb(MYELIN_COLOR), 1.0)
    return rgba


def annotate_instances(ax, lbl: np.ndarray):
    """Escribe sobre cada fibra su valor de axón X / mielina X+1, como en el formato del challenge."""
    fid = np.where(lbl >= 2, lbl // 2, 0)
    for f, sl in enumerate(ndi.find_objects(fid), start=1):
        if sl is None:
            continue
        m = lbl[sl] == 2 * f
        if not m.any():
            m = fid[sl] == f
        d = ndi.distance_transform_edt(m)  # punto más interno de la máscara
        y, x = np.unravel_index(np.argmax(d), d.shape)
        ax.text(x + sl[1].start, y + sl[0].start, f"{2 * f}/{2 * f + 1}", ha="center", va="center",
                fontsize=LABEL_FONTSIZE, weight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.15", fc="black", alpha=0.6, lw=0))


def show_prediction(img: np.ndarray, lbl: np.ndarray, title: str):
    n_fib, g_med = fiber_stats(lbl)
    fig, axes = plt.subplots(1, 3, figsize=(21, 5.6), sharex=True, sharey=True, layout="constrained")
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title("Original image")
    axes[1].imshow(instances_to_rgb(lbl), interpolation="nearest")
    annotate_instances(axes[1], lbl)
    axes[1].set_title(f"Instances: axon X / myelin X+1 ({n_fib} fibers)")
    axes[2].imshow(img, cmap="gray")
    axes[2].imshow(overlay_rgba(lbl), interpolation="nearest")
    axes[2].set_title("Overlay (axon and myelin boundaries)")
    for ax in axes:
        ax.axis("off")
    handles = [Patch(color=AXON_COLOR, label="Axon (X, even)"), Patch(color=MYELIN_COLOR, label="Myelin (X+1, odd)"),
               Patch(color="#999999", label="Middle panel: one color per fiber (light = axon X, dark = myelin X+1)")]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
    fig.suptitle(f"{title}  ·  median g-ratio {g_med:.3f}", weight="semibold")
    return fig

## 7. Load the 10 models

In [ ]:
torch.manual_seed(0)
predictors = []
for m in MODEL_DIRS:
    pred = nnUNetPredictor(tile_step_size=STEP, use_mirroring=True, device=device, allow_tqdm=False)
    pred.initialize_from_trained_model_folder(str(m), FOLDS, CHECKPOINT)
    predictors.append(pred)
    print(f"{m}: {len(pred.list_of_parameters)} folds cargados")

## 8. Inference

Each mask is saved as soon as its image is predicted, and the image is shown next to its prediction.

In [ ]:
if SAVE_FIGURES:
    (OUTPUT_DIR / "figures").mkdir(exist_ok=True)

rows = []
t_start = time.time()
for i, p in enumerate(files, 1):
    t0 = time.time()
    img = read_image(p)
    small = downsample(img)[None, None].astype(np.float32)  # (c=1, z=1, H, W)
    probs = None
    for pred in predictors:  # promedio con el mismo peso por carpeta de modelo
        _, pr = pred.predict_single_npy_array(small, {"spacing": (999, 1, 1)}, None, None, True)
        probs = pr if probs is None else probs + pr
    lbl = probs_to_label(probs / len(predictors), img.shape)
    write_label(OUTPUT_DIR / p.name, lbl)
    rows.append(dict(image=p.name, fibers=int(lbl.max() // 2), seconds=round(time.time() - t0, 1)))
    print(f"[{i}/{len(files)}] {p.name}: {rows[-1]['fibers']} fibras, {rows[-1]['seconds']} s "
          f"-> {OUTPUT_DIR / p.name}")
    if SHOW_FIGURES or SAVE_FIGURES:
        fig = show_prediction(img, lbl, f"{p.name} ({i}/{len(files)})")
        if SAVE_FIGURES:
            fig.savefig(OUTPUT_DIR / "figures" / f"{p.stem}.png", dpi=100)
        if SHOW_FIGURES:
            display(fig)
        plt.close(fig)

summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(f"total: {summary.fibers.sum()} fibras en {len(summary)} imágenes, {(time.time() - t_start) / 60:.1f} min")
summary

## 9. Check the output format

In [ ]:
problems = []
for p in files:
    lbl = tifffile.imread(OUTPUT_DIR / p.name)
    ids = set(np.unique(lbl).tolist()) - {0}
    axons = {v for v in ids if v % 2 == 0}
    myelins = {v for v in ids if v % 2 == 1}
    if lbl.dtype != np.uint16 or lbl.shape != read_image(p).shape:
        problems.append(f"{p.name}: dtype/shape {lbl.dtype} {lbl.shape}")
    if 1 in ids:
        problems.append(f"{p.name}: contiene el valor 1")
    if {v + 1 for v in axons} != myelins:
        problems.append(f"{p.name}: pares X / X+1 incompletos")
print(f"OK: las {len(files)} predicciones son uint16, del tamaño de la imagen y con pares X / X+1 completos"
      if not problems else "\n".join(problems))
print(f"predicciones en: {OUTPUT_DIR.resolve()}")